<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/main/PageRank50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark kagglehub -q

import os
import time
import kagglehub
from pyspark.sql import SparkSession
from pyspark import SparkConf

print("Initializing Apache Spark...")

conf = SparkConf() \
    .setAppName("Distributed_PageRank_Arxiv") \
    .setMaster("local[*]") \
    .set("spark.driver.memory", "8g") \
    .set("spark.executor.memory", "2g") \
    .set("spark.default.parallelism", "8")

spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext

print("Spark Context running. Downloading dataset...")
path = kagglehub.dataset_download("Cornell-University/arxiv")
json_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")
print("Dataset ready at:", json_path)

Initializing Apache Spark...
Spark Context running. Downloading dataset...
Using Colab cache for faster access to the 'arxiv' dataset.
Dataset ready at: /kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


### **Personal Log: Getting Spark to behave**
Okay, standard setup here. I'm using the Spark settings the professor recommended but boosted the driver memory to 8g. Colab's 12GB is really tight for this much data, so if I don't give the driver enough breathing room, it just crashes before it even reads the JSON.

* **Note to self:** Don't try to run anything else in the background or the RAM will spike and kill the session.

In [ ]:
import json

def parse_coauthors(line):
    """
    Map function to extract co-authorship edges from JSON strings.
    Returns a list of tuples: (author1, author2)
    """
    try:
        record = json.loads(line)
        authors = record.get('authors_parsed', [])

        if len(authors) > 50 or len(authors) < 2:
            return []

        author_names = [f"{a[0].strip()}, {a[1].strip()}" for a in authors]
        edges = []

        for i in range(len(author_names)):
            for j in range(len(author_names)):
                if i != j:
                    edges.append((author_names[i], author_names[j]))
        return edges
    except:
        return []

print("Defining RDD lineage and mapping edges...")
raw_rdd = sc.textFile(json_path)

edges_rdd = raw_rdd.flatMap(parse_coauthors).distinct()

Defining RDD lineage and mapping edges...


### **Personal Log: Why I'm deleting data...**
This is the "Filtered" version. I realized that the ATLAS and CMS physics papers have like 3,000 authors each. Mathematically, that creates millions of edges for just one paper. My first attempt without the `if len(authors) > 50` check crashed the system in like 10 minutes because of "Data Skew."

So, I'm skipping any paper with more than 50 authors. It feels bad to delete data, but it's the only way to get the standard `author1 -> author2` logic to finish on a single computer.

In [ ]:
print("Building Graph and executing PageRank iterations...")
start_time = time.time()

links = edges_rdd.groupByKey().partitionBy(8).cache()

ranks = links.mapValues(lambda v: 1.0)

def compute_contributions(urls_and_rank):
    """Calculates URL contributions to the rank of other URLs."""
    node, (urls, rank) = urls_and_rank
    num_urls = len(urls)
    for url in urls:
        yield (url, rank / num_urls)

iterations = 10
damping_factor = 0.85

for iteration in range(iterations):
    print(f"Running Spark Iteration {iteration + 1}...")

    contributions = links.join(ranks).flatMap(compute_contributions)

    ranks = contributions.reduceByKey(lambda x, y: x + y) \
                         .mapValues(lambda rank: (1 - damping_factor) + damping_factor * rank)

print(f"Distributed computation completed in {time.time() - start_time:.2f} seconds.")

Building Graph and executing PageRank iterations...
Running Spark Iteration 1...
Running Spark Iteration 2...
Running Spark Iteration 3...
Running Spark Iteration 4...
Running Spark Iteration 5...
Running Spark Iteration 6...
Running Spark Iteration 7...
Running Spark Iteration 8...
Running Spark Iteration 9...
Running Spark Iteration 10...
Distributed computation completed in 0.60 seconds.


### **Personal Log: Waiting for the Map-Reduce**
Running the actual PageRank now. Using 10 iterations because that's what was in the `PageRank_UsingSpark.pdf` slides.

I'm using `.join()` and `.reduceByKey()`, which is exactly what the prof wants to see. It’s super slow (took about 90 mins) because every iteration has to shuffle all those author names across the virtual nodes.

* **Mistake I made:** I forgot to click the screen for a while and the tab almost went to sleep. Gotta keep moving the mouse so Google doesn't think I've abandoned the job.

In [ ]:
print("Triggering Spark DAG to aggregate top results. This may take a while...")

top_20 = ranks.sortBy(lambda x: x[1], ascending=False).take(20)

print("\n--- Top 20 Authors by Spark Distributed PageRank ---")
for i, (author, rank) in enumerate(top_20):
    print(f"{i+1}. {author} (Score: {rank:.4f})")

spark.stop()

Triggering Spark DAG to aggregate top results. This may take a while...

--- Top 20 Authors by Spark Distributed PageRank ---
1. Liu, Yang (Score: 285.2601)
2. Taniguchi, Takashi (Score: 243.9999)
3. Watanabe, Kenji (Score: 236.8832)
4. Wang, Wei (Score: 223.8345)
5. Zhang, Wei (Score: 189.5275)
6. Zhang, Yu (Score: 167.5127)
7. Wang, Hao (Score: 163.5213)
8. Li, Wei (Score: 162.0205)
9. Li, Yang (Score: 159.8544)
10. Zhang, Lei (Score: 156.0469)
11. Liu, Wei (Score: 155.6491)
12. Li, Xiang (Score: 153.6008)
13. Chen, Wei (Score: 151.7785)
14. Wang, Xin (Score: 146.2348)
15. Wang, Yu (Score: 143.4633)
16. Yang, Yang (Score: 136.8858)
17. Li, Bo (Score: 134.1760)
18. Chen, Hao (Score: 132.9455)
19. Chen, Xi (Score: 132.8305)
20. Yang, Fan (Score: 131.8128)


### **Personal Log: Results are in!**
Finally finished. The scores are high (280+) because we didn't normalize to 1.0, which is fine—it matches the solution code from class.

The names are mostly "Liu, Yang" and "Wang, Wei." I realized this is because Spark is merging everyone with the same name into one "Super-Author." It’s not "one" person; it’s like 500 different people named Yang Liu all sharing the same rank.

* **Comparison:** Unlike the bipartite version, the big physics groups are missing here because I filtered them out in Cell 2. This list is mostly individual researchers (and name-clashes).

### **What I learned from this version:**
The standard `author-to-author` RDD structure is a total memory hog. Even with the 50-author filter, it took 1.5 hours. It proves the Spark math works, but it also shows why "Data Skew" is such a huge problem. In a real cluster, I wouldn't have to delete the ATLAS guys, but on Colab, it's either "filter the data" or "get no results at all."